# Clustering the result of a MCA with DBSCAN

NOT yet finished, do not use !

#### Documentation 

[Comparing clustering algorithms](https://hdbscan.readthedocs.io/en/latest/comparing_clustering_algorithms.html) 


In [ ]:
import numpy as np
import pandas as pd

import sqlite3 as sql

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import seaborn as sns

import networkx as nx
import itertools

import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import fcluster
from scipy.spatial.distance import cdist


from sklearn.metrics import silhouette_score


from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

from prince import MCA


In [ ]:
from sklearn.cluster import HDBSCAN

In [ ]:
### this library allows to execute SQL queries on dataframes
# and even joins
import duckdb

In [ ]:
### Importer un module de fonctions crées ad hoc
##  ATTENTION : le fichier 'sparql_functions.py' doit se trouver 
#   dans un dossier qui se situe dans le chemin ('path') de recherche
#   vu par le présent carnet Jupyter afin que
#   l'importation fonctionne correctement

import sys
from importlib import reload


# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')



In [ ]:
## these are local libraries of functions 
import bivariate_library as bl
import correspondence_analysis_library as cal
import cluster_functions as cf


In [ ]:
### Use this command to reload the functions 
# if you modify them
print(reload(cf))


In [ ]:
import warnings
warnings.filterwarnings('ignore')


## Create a dataframe with the data to be analysed

In this notebook, we use the data produced in the [multiple correspondence analysis (MCA) notebook ](da5_MCA.ipynb), which was prepared and exported into this file [this file](da_data/da5-MCA-clusters.csv) after the sparse data was aggregated.




In [ ]:
file_address='da_data/da5-MCA-clusters.csv'
df_pm = pd.read_csv(file_address)
df_pm.head(3)

In [ ]:
print(df_pm.columns.to_list())

In [ ]:
### Rename columns to have shorter labels
df_pm.columns=['person_uri',
 'labelPer',
 'birthYear',
 'gender',
 'labelPlace',
 'geometry',
 'REGION',
 'NAME_ENGL',
 'country',
 'per_activ',
 'pk_person_features',
 'occup_main',
 'occup_sec',
 'empl']

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_pm.info()

In [ ]:
### change display options for jupyter notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Reset to default settings if needed later
# pd.reset_option('display.max_columns')

## Factor analysis (MCA)

We only consider for MCA the coded features of persons, transformed into qualitative categories 

We don't use the periods for defining profiles, our research question is about inspecting the correlation between profiles and periods, and identify specific profiles for specific periods

In [ ]:
### Define the variables that will be used for analysing the population profiles
df_actives = df_pm[['gender', 'country', 'occup_sec', 'empl']].copy(deep=True)
print(df_actives.columns)
df_actives.head(2)

In [ ]:
### We add here also the main occupation, hoping it helps to get more structure
df_actives = df_pm[['gender', 'country', 'occup_main', 'occup_sec', 'empl']].copy(deep=True)
print(df_actives.columns)
df_actives.head(2)

### Process in two steps

First fit, then transform.

The mca variable will be used to project illustrative variables, in our case periods of activity

In [ ]:
### MCA with benzecri correction: whole intertia in 5 axis
# Equivalent to fit_transform but in to steps
mca = MCA(n_components=10, n_iter=10, correction='benzecri',random_state=42)
mca.fit(df_actives)

In [ ]:
### Benzecri correction and approximation allows to summarize 
# the most relevant aspects of the variance in five axes (in this case)
mca.eigenvalues_summary

In [ ]:
### Coordinates of categories, i.e. columns in the complete disjoint table
print(len(mca.column_coordinates(df_actives)))
dfcc = mca.column_coordinates(df_actives)
dfcc.head()

In [ ]:
### Coordinates of persons (rows)
# The same result as using the method 'fit_transform'
print(len(mca.row_coordinates(df_actives)))
dfrc = mca.row_coordinates(df_actives)
dfrc.head()

In [ ]:
### Equivalent to fit transform in one step
X_mca=mca.transform(df_actives)
X_mca.head()

### Prepare first two axis

In [ ]:
ax1=mca.eigenvalues_summary.iloc[0, 1]
ax2=mca.eigenvalues_summary.iloc[1, 1]

### Calculate position of illustrative variable 'per_activ'

We don't use the periods for defining profiles, our research question is about inspecting the correlation between profiles and periods, and identify specific profiles for specific periods

In [ ]:
df_illustrative = df_pm[['per_activ']].copy(deep=True)
# df_illustrative.head(2)

In [ ]:
df_per_activ=pd.DataFrame(df_pm.groupby('per_activ').size())
# df_per_activ.index=df_per_activ.index.map(lambda x: f'per_activ__{x}')
df_per_activ.columns=['number']
df_per_activ

In [ ]:
### This script situates the illustrative variable 'per_activ'
# at the mean position in the geometric space using the coordinates 
# of all the individuals

row_coords = mca.row_coordinates(df_actives)

# 3. Select the single illustrative variable column
illus_var = df_illustrative['per_activ'] # Replace with actual column name

# 4. Calculate position for each category in that variable
supp_positions = {}

for category in illus_var.dropna().unique():
    # Find indices of individuals who have this category
    mask = illus_var == category
    
    # Calculate the barycenter (mean) of their row coordinates
    # This is the position of the supplementary category
    centroid = row_coords[mask].mean(axis=0)
    
    supp_positions[category] = centroid.values

# 5. Convert to DataFrame for easy use/plotting
df_per_activ_pos = pd.DataFrame(
    supp_positions, 
    index=[i for i in range(row_coords.shape[1])]
).T  # Transpose so rows are categories, columns are dimensions

df_per_activ_pos.head(2)

In [ ]:
### We add the label and the number of individuals to the mean position
df_per_activ=df_per_activ.join(df_per_activ_pos)
df_per_activ

In [ ]:
### dimensions
d1=1 # 0
d2=2 # 1

# Create the plot
fig, ax = plt.subplots(figsize=(20, 20))

# Plot Active Variables (Blue/Grey default)
ax.scatter(dfcc[d1], dfcc[d2], s=30, alpha=0.7, c='blue', label='Active Variables', zorder=3)

# Label Active Variables
for label, x, y in zip(dfcc.index, dfcc[d1], dfcc[d2]):
    ax.annotate(label, (x, y), fontsize=10, ha='right', va='bottom', color='blue')

# Plot Illustrative Variables (Red)
if not df_per_activ.empty:
    ax.scatter(df_per_activ[d1], df_per_activ[d2], 
               s=30, alpha=0.8, c='red', marker='s', label='Illustrative Variables', zorder=4)
    
    # Label Illustrative Variables (optional: adjust font size/color to distinguish)
    for label, x, y in zip(df_per_activ.index, df_per_activ[d1], df_per_activ[d2]):
        ax.annotate(label, (x, y), fontsize=10, ha='left', va='top', color='red', fontweight='bold')

# Add reference lines
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

# Set labels and title
ax.set_xlabel(f"Component 1 ({ax1})") # Formatting percentage if applicable
ax.set_ylabel(f"Component 2 ({ax2})")
ax.set_title("MCA – Axis 1 & 2 (Active vs Illustrative)")

# Add legend to distinguish groups
ax.legend(loc='best')

# Save the figure
img_address='doc_images/mca_prince_with_periods.png'
plt.savefig(img_address, dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## HDBSCAN


Cluster with the DBSCAN method

In [ ]:

# Optional: Scale coordinates (often helps DBSCAN)
scaler = StandardScaler()
mca_scaled = scaler.fit_transform(X_mca)
print(mca_scaled[:2])


### Prepare hyperparameters

### Apply HDBSCAN

Distance Metric: Use Euclidean distance on the selected MCA dimensions. This effectively approximates the Chi-square distance used in categorical analysis.

In [ ]:
# Candidate eps from the graph
min_cluster_size = 20
min_samples = 5 # The minimum number of points to form a dense region.


# ==========================================
# 2. RUN HDBSCAN
# ==========================================
# Key Parameters:
# min_cluster_size: Similar to DBSCAN's min_samples. 
#                   Rule of thumb: >= Number of MCA dimensions (e.g., 5-10).
# min_samples: Controls conservatism. Higher = more points labeled as noise.
#              If None, defaults to min_cluster_size.
# cluster_selection_epsilon: Optional. Max distance for points to join a cluster.

clusterer = HDBSCAN(
    min_cluster_size=min_cluster_size,      # Adjust based on your data size
    min_samples=min_samples,            # Helps filter noise (optional, can be None)
    metric='euclidean'       # MCA coordinates are Euclidean
)

clusters=clusterer.fit_predict(mca_scaled)


# Add cluster label (number) to original dataframe
df_actives['cluster'] = clusters
df_pm['cluster'] = clusters

cluster_counts=df_actives['cluster'].value_counts()
print(len(cluster_counts))

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    print(cluster_counts)

In [ ]:

# ==========================================
# 4. CALCULATE MEDOIDS & EXTRACT LABELS
# ==========================================
def get_cluster_summary(df_original, mca_coords, labels):
    """
    Returns a DataFrame with:
    - Cluster ID
    - Size of cluster
    - Medoid Index (original row index)
    - All original categorical columns of the Medoid
    """
    summary_data = []
    
    unique_labels = set(labels)
    # Remove noise (-1) if you don't want it in the summary
    # If you want to analyze noise, keep it but note it has no true medoid
    ## cluster_ids = sorted([l for l in unique_labels if l != -1])
    ### keep the medoid
    cluster_ids = sorted([l for l in unique_labels])
    
    for cid in cluster_ids:
        # Get indices of points in this cluster
        cluster_mask = (labels == cid)
        cluster_indices = np.where(cluster_mask)[0]
        
        if len(cluster_indices) == 0:
            continue
            
        # Get MCA coordinates for this cluster
        cluster_coords = mca_coords.iloc[cluster_indices]
        
        # Calculate pairwise distances
        dist_matrix = pairwise_distances(cluster_coords)
        
        # Find medoid: point with minimum sum of distances to others
        sum_dists = dist_matrix.sum(axis=1)
        medoid_local_idx = np.argmin(sum_dists)
        medoid_original_idx = cluster_indices[medoid_local_idx]
        
        # Extract the original categorical labels for the medoid
        medoid_row = df_original.iloc[medoid_original_idx]
        
        # Build summary row
        summary_row = {
            'Cluster_ID': cid,
            'Cluster_Size': len(cluster_indices),
            'Medoid_Original_Index': medoid_original_idx
        }
        
        # Add all original categorical columns (excluding 'Cluster_ID' if already added)
        for col in df_original.columns:
            if col != 'Cluster_ID' and  col != 'cluster':
                # summary_row[f'Medoid_{col}'] = medoid_row[col]
                summary_row[f'{col}'] = medoid_row[col]
                
        summary_data.append(summary_row)
    
    return pd.DataFrame(summary_data)


In [ ]:
### Generate the summary table with original labels and medoids
# we use the function defined above
cluster_summary = get_cluster_summary(df_actives, X_mca, clusters)

In [ ]:
cluster_summary.sort_values('Cluster_Size',ascending=False).head()

In [ ]:
print(cluster_summary.columns.to_list())

In [ ]:
### rename columns
cluster_summary.columns=['cluster',
 'Cluster_Size',
 'Original_Index',
 'gender',
 'country',
 'occup',
 'empl']
cluster_summary.head(2)

In [ ]:
dfrc.head(2)

In [ ]:
### coordinates of medoids as cluster profiles
cluster_profiles=cluster_summary[['cluster',
 'Cluster_Size',
 'Original_Index',
 'gender',
 'country',
 'occup',
 'empl']].merge(dfrc, left_on='Original_Index', right_index=True)

In [ ]:
cluster_profiles.head(2)

In [ ]:
### Create a label with the mode categories
cluster_profiles['label'] = cluster_profiles.apply(
    lambda x: str(x.name) + "_" + "_".join(x[['gender', 'country', 'occup', 'empl']].astype(str)), 
    axis=1
)
cluster_profiles.head(1)

In [ ]:
### Create a label with the mode categories

cluster_profiles[cluster_profiles['cluster']==-1]

In [ ]:
### Inspect the already created illustrative variable 'per_act'
df_per_activ

### Plot the cluster profiles

In [ ]:
### Get the inertia proportion for the first two axis
ax1=mca.eigenvalues_summary.iloc[0, 1]
ax2=mca.eigenvalues_summary.iloc[1, 1]

In [ ]:

# Get coordinates for illustrative variables
# Note: Ensure 'df_illustrative' contains the column names of your illustrative variables
df_illust_coords = mca.column_coordinates(df_illustrative)

# Create the plot
fig, ax = plt.subplots(figsize=(20, 20))

# Plot Active Variables (Blue/Grey default)
ax.scatter(cluster_profiles[0], cluster_profiles[1], 
           s=cluster_profiles['Cluster_Size'], alpha=0.7, c='blue', label='Active Variables', zorder=3)

# Label Active Variables
for label, x, y in zip(cluster_profiles.label, cluster_profiles[0], cluster_profiles[1]):
    ax.annotate(label, (x, y), fontsize=10, ha='right', va='bottom', color='blue')

# Plot Illustrative Variables (Red)
if not df_per_activ.empty:
    ax.scatter(df_per_activ[1], df_per_activ[2], 
               s=df_per_activ['number']/5, alpha=0.8, c='red', marker='s', label='Illustrative Variables', zorder=4)
    
    # Label Illustrative Variables (optional: adjust font size/color to distinguish)
    for label, x, y in zip(df_per_activ.index, df_per_activ[1], df_per_activ[2]):
        ax.annotate(label, (x, y), fontsize=10, ha='left', va='top', color='red', fontweight='bold')

# Add reference lines
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

# Set labels and title
ax.set_xlabel(f"Component 1 ({ax1})") # Formatting percentage if applicable
ax.set_ylabel(f"Component 2 ({ax2})")
ax.set_title("MCA – Axis 1 & 2 (Active vs Illustrative)")

# Add legend to distinguish groups
ax.legend(loc='best')

# Save the figure
img_address='doc_images/dbscan_with_clusters.png'
plt.savefig(img_address, dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

In [ ]:
cluster_profiles.head(1)

#### Inspect frequency of individuals identical with centroid

In [ ]:
### this dataframe contains only rows that are identical with the centroid in each cluster
merged_df = df_actives.merge(cluster_profiles, on=['gender', 'country', 'occup', 'empl', 'cluster'], how='inner')

In [ ]:
### Persons with medoid value in proportion to population

# only persons identical to centroids of their cluster
print(len(merged_df))

# all the persons
print(len(df_pm))


# frequency
print(str(round((len(merged_df)/len(df_pm)*100),1))+' %')


#### Inspect with identical cluster profile (modes)

In [ ]:
df_actives.head(1)

In [ ]:
variables=['gender', 'country', 'occup', 'empl']

In [ ]:
def get_joint_mode_profile(df, cluster_col, variables):
    """
    Finds the most frequent combination of variables (joint mode) for each cluster.
    Returns the combination values and their frequency count.
    """
    results = []
    
    # Group by Cluster ID
    grouped = df.groupby(cluster_col)
    
    for cluster_id, group in grouped:
        # 1. Create a temporary view of just the relevant variables
        subset = group[variables].copy()
        
        # 2. Count occurrences of each unique combination
        # groupby.size() counts rows for each unique combination of the listed columns
        combo_counts = subset.groupby(variables).size().reset_index(name='Count')
        
        # 3. Find the row with the maximum count
        if not combo_counts.empty:
            # idxmax() returns the index of the maximum value
            max_idx = combo_counts['Count'].idxmax()
            best_combo = combo_counts.loc[max_idx]
            
            # 4. Build the result dictionary
            row = {
                'cluster': cluster_id,
                'cluster_size': len(group),
                'joint_mode_count': best_combo['Count']
            }
            
            # Add the variable values from the best combination
            for var in variables:
                row[f'{var}'] = best_combo[var]
            
            # Calculate Purity: What % of the cluster shares this exact profile?
            row['purity_ratio'] = best_combo['Count'] / len(group)
            
            results.append(row)
        else:
            # Handle empty clusters (shouldn't happen with valid DBSCAN output)
            pass
            
    return pd.DataFrame(results)


In [ ]:

# Execute the function
joint_mode_df = get_joint_mode_profile(df_actives, 'cluster', variables)

# Display results
#print("--- Most Frequent Combination (Joint Mode) per Cluster ---")
#print(joint_mode_df.to_string())

# Optional: Sort by Purity to see which clusters are most homogeneous
# print(joint_mode_df.sort_values('Purity_Ratio', ascending=False))

In [ ]:
joint_mode_df

In [ ]:
### this dataframe contains only rows that are identical with the centroid in each cluster
merged_df_prof = df_actives.merge(joint_mode_df, on=['gender', 'country', 'occup', 'empl', 'cluster'], how='inner')

In [ ]:
### Persons with medoid value in proportion to population

# only persons identical to centroids of their cluster
print(len(merged_df_prof))

# all the persons
print(len(df_pm))
cluster_profiles

# frequency
print(str(round((len(merged_df_prof)/len(df_pm)*100),1))+' %')


#### Keep the clu_prof 

We keep the first methodology to define the cluster center, based on modes.

But we observe from different runs for 16,32 and 64 cluster, that smalle clusters tend to be nearer to the mean profile (vs the mode)

In [ ]:
clu_prof.head()

In [ ]:
print(len(merged_df_prof))
merged_df_prof.head(2)

In [ ]:
### Count per cluster how many rows have same values as centroid
dfcen = pd.DataFrame(merged_df_prof.groupby(by=['cluster']).size())
dfcen.columns=['n_cluster']
dfcen.sort_values(by='n_cluster', ascending=False).head()

In [ ]:
### How many rows in total have gender:female
dfgf = pd.DataFrame(df_actives[df_actives.gender=='female'].groupby(by=['cluster']).size())
dfgf.columns=['number_f']
(print(dfgf.sort_values(by='number_f', ascending=False).iloc[:30]))

In [ ]:
### Add cluster to df_pm
df_pm=df_pm.join(df_actives, rsuffix='_r')

In [ ]:
df_pm.head(2)

In [ ]:
### Group and count countries in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'country_r']).size()).reset_index()
result_df.columns=['cluster','country_r','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])
result_df.head()


# aggregated countries per cluster
dfg_country = result_df.groupby('cluster')[['country_r', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_country['countries_list'] = dfg_country.aggregated_data.apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_country=dfg_country.drop(columns=['cluster','aggregated_data'])


with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_country.head())

In [ ]:
### Group and count occupations in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'occup']).size()).reset_index()
result_df.columns=['cluster','occup','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])


# aggregated occupations per cluster
dfg_occupation = result_df.groupby('cluster')[['occup', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_occupation['occ_list'] = dfg_occupation['aggregated_data'].apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_occupation=dfg_occupation.drop(columns=['cluster','aggregated_data'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_occupation.head())


In [ ]:
### Group and count employer classes in clusters

# countries per cluster
result_df = pd.DataFrame(df_pm.groupby(by=['cluster', 'empl']).size()).reset_index()
result_df.columns=['cluster','empl','number']
result_df=result_df.sort_values(['cluster','number'], ascending=[True,False])


# aggregated occupations per cluster
dfg_empl = result_df.groupby('cluster')[['empl', 'number']].apply(
    lambda x: x.values.tolist()
).reset_index(name='aggregated_data')

### Take the first five per cluster
dfg_empl['empl_list'] = dfg_empl['aggregated_data'].apply(
    lambda lst: ', '.join([f"{item[0]}: {item[1]}" for item in lst[:5]])
)
dfg_empl=dfg_empl.drop(columns=['cluster','aggregated_data'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(dfg_empl.head())


#### Join and inspect

Aggregating the former tables and counts allows to inspect the clusters and identifying the structuring elements of the clusters 

In [ ]:
clu_prof_short=clu_prof.iloc[:,[1,2,3,4,5,11]]

In [ ]:
print(len(clu_prof_short))
clu_prof_short

In [ ]:
### Join the different dataframes using the default joining on indexes
# If an error appears in joining, restart the whole process from cenroid_df onward


clu_prof_short=clu_prof_short.join(dfcen)
clu_prof_short.rename(columns={'n_cluster': 'number_centroid'}, inplace=True)
clu_prof_short=clu_prof_short.join(dfgf).fillna(0)
clu_prof_short['number_f'] = clu_prof_short['number_f'].astype(int)
clu_prof_short['prop_f'] = clu_prof_short.apply(lambda x: (x['number_f']/x['number_in_cl']), axis=1)
clu_prof_short['prop_f']=clu_prof_short['prop_f'].round(2)
clu_prof_short=clu_prof_short.join(dfg_country)
clu_prof_short=clu_prof_short.join(dfg_occupation)
clu_prof_short=clu_prof_short.join(dfg_empl)

In [ ]:
clu_prof_short.number_centroid=clu_prof_short.number_centroid.astype('int')

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(clu_prof_short.sort_values(by='number_in_cl', ascending=False).style.set_properties(subset=['occ_list', 'countries_list'], **{'text-align': 'left'}))

## Test if relation between clusters and periods

In [ ]:
# Validate against Generation (Illustrative Variable)
# Cross-tabulation : contingency_table

df = df_pm

observed = pd.crosstab(df['cluster'], df['per_activ'])


In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.108
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(8,8))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=3,num_y_axis=4,figsize=(8,8))

In [ ]:
### Number of clusters
width= df['cluster'].max()
pp = bl.plot_chi2_residuals(observed.T, figsize=(width, 9))

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(pd.DataFrame(clu_prof_short[['label','number_in_cl','number_centroid','number_f', 'prop_f']]))

In [ ]:
### write centroids to database table for more deep inspection
db = '../../data/data_analysis.db'

#table_name='clusters_kmodes_centroids_c54'
table_name='mca_kmeans_clusters_centroids'

#clu_prof_short['run']='cen16'
clu_prof_short['run']='cen32'
#clu_prof_short['run']='cen64'


conn = sql.connect(db)
# commented for safety, uncomment to execute
# clu_prof_short.to_sql(table_name, conn, if_exists='append', index=True)
conn.close()

In [ ]:
pers_cluster=df_pm[['person_uri', 'cluster']]
pers_cluster.head()

In [ ]:
### write clusters to database table for more deep inspection
db = '../../data/data_analysis.db'

table_name='mca_kmeans_clusters'

# pers_cluster['run']='cen16'
pers_cluster['run']='cen32'
# pers_cluster['run']='cen64'

conn = sql.connect(db)
# cursor = conn.cursor()
pers_cluster.to_sql(table_name, conn, if_exists='append', index=True)
conn.close()

## Plot clusters

In [ ]:
categorical_cols = df_actives.columns.to_list()[:4]
print(categorical_cols)

In [ ]:
df_actives.head(1)

In [ ]:
## ici on voit les attractions de modalités
# les variables au centre ou très liées structurent le cluster
pict_address='images/kmeans_ACM_clusters_4_variables_32cl.png'
cf.plot_cluster_networks(df_actives, categorical_cols, 46, 
        pict_address=pict_address)

## Test clusters correspondence

In [ ]:
conn = sql.connect(db)


# 2. Define your SQL query
query = """
SELECT mkc.person_uri, mkc.cluster cluster_kmean, kc.cluster cluster_kmode
from mca_kmeans_clusters mkc, kmodes_clusters kc 
where mkc.person_uri = kc.person_uri
and mkc.run = 'cen32'
and kc.run = 'cen32'
order by cluster_kmean, cluster_kmode
"""

# 3. Execute the query and load results into a DataFrame
dfq = pd.read_sql_query(query, conn)

# 4. Close the connection
conn.close()

# Display the first few rows
print(dfq.head())

In [ ]:
# 5. Validate against Generation (Unused Variable)
# Cross-tabulation : contingency_table
observed = pd.crosstab(dfq['cluster_kmode'], dfq['cluster_kmean'])


In [ ]:
bl.check_chi_square_test_validity(observed)

In [ ]:
expected=bl.bivariate_stats(observed)

### CA

In [ ]:
afc = fa.CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.108
cal.print_eigenvalue(afc)

In [ ]:
cal.dim_contributions(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(8,8))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=3,num_y_axis=4,figsize=(8,8))

In [ ]:
pp = bl.plot_chi2_residuals(observed.T, figsize=(15, 15))

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(pd.DataFrame(clu_prof_short[['label','number_in_cl','number_centroid','number_f', 'prop_f']]))